In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
import os

In [2]:
from langchain_openai import ChatOpenAI
model=ChatOpenAI(model='gpt-4o')
output=model.invoke("hi")
print(output.content)

Hello! How can I assist you today?


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")
len(embeddings.embed_query("hi"))

384

In [4]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\DELL\AppData\Local\Temp\ipykernel_5920\4068926105.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


In [5]:
loader=DirectoryLoader("./data2",glob="./*.txt",loader_cls=TextLoader)

In [6]:
docs=loader.load()

In [7]:
docs

[Document(metadata={'source': 'data2\\usa.txt'}, page_content="ðŸ‡ºðŸ‡¸ Overview of the U.S. Economy\nThe United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.\n\nU.S. GDP â€“ Size, Composition, and Global Share\nAs of 2024, the United Statesâ€™ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 in the world by nominal GDP, far ahead of China (wh

In [8]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

In [9]:
new_docs=text_splitter.split_documents(documents=docs)

In [11]:
len(new_docs)

56

In [12]:
new_docs

[Document(metadata={'source': 'data2\\usa.txt'}, page_content='ðŸ‡ºðŸ‡¸ Overview of the U.S. Economy'),
 Document(metadata={'source': 'data2\\usa.txt'}, page_content='The United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy,'),
 Document(metadata={'source': 'data2\\usa.txt'}, page_content='It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a'),
 Document(metadata={'source': 'data2\\usa.txt'}, page_content='a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services.'),
 Document(metadata={'source': 'data2\\usa.txt'}, page_content='innovation, global trade, and financial

In [13]:
doc_strings = [doc.page_content for doc in new_docs]

In [14]:
doc_strings

['ðŸ‡ºðŸ‡¸ Overview of the U.S. Economy',
 'The United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy,',
 'It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a',
 'a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services.',
 'innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.',
 'U.S. GDP â€“ Size, Composition, and Global Share',
 'As of 2024, the United Statesâ€™ nominal GDP is estimated to be around $28 trillion USD, accounting for appr

In [15]:
db=Chroma.from_documents(new_docs,embeddings)
retriever=db.as_retriever(search_kwargs={"k": 3})

In [16]:
retriever.invoke("industrial growth of usa?")

[Document(metadata={'source': 'data2\\usa.txt'}, page_content='Looking forward, the U.S. economy is expected to grow at a moderate pace, powered by innovation in AI, green energy, robotics, biotech, and quantum computing. The Biden administrationâ€™s Inflation'),
 Document(metadata={'source': 'data2\\usa.txt'}, page_content='ðŸ‡ºðŸ‡¸ Overview of the U.S. Economy'),
 Document(metadata={'source': 'data2\\usa.txt'}, page_content='The U.S. economy remains the engine of global growth, backed by unmatched innovation, financial dominance, and a strong institutional framework. Its $28 trillion GDP and influence over global')]

# Creation of pydantic class

In [19]:
import operator
from typing import List
from pydantic import BaseModel , Field
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph,END
from langchain_core.output_parsers import PydanticOutputParser

In [18]:
class TopicSelectionParser(BaseModel):
    Topic:str=Field(description="selected topic")
    Reasoning:str=Field(description='Reasoning behind topic selection')

In [20]:
parser = PydanticOutputParser(pydantic_object=TopicSelectionParser)

In [22]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"Topic": {"description": "selected topic", "title": "Topic", "type": "string"}, "Reasoning": {"description": "Reasoning behind topic selection", "title": "Reasoning", "type": "string"}}, "required": ["Topic", "Reasoning"]}\n```'

### Different components of Langgraph

1. State
   . The Memory of your application
   Share data structure that persistance accross all steps
   * Can contain message, variable, flags or any data
  
2. Nodes

   * Individual functions or operations in your workflow
   * Each Nodes receives the current state and returns the updates
   * Think of them as "workers" that do specific tasks

3. Edges

   * Connections between the Nodes that define the flow
   * can be simple edge (A->B) or conditional (A->B or C based on the condition)

4. Graph

   * The ccomplete workflow combining nodes and edges
   * 

### this below agentstate is just for the explnation like how state works

In [23]:
Agentstate={}

In [24]:
Agentstate["messages"]=[]
Agentstate["number"] = 0

In [25]:
Agentstate

{'messages': [], 'number': 0}

In [26]:
Agentstate["number"] += 1

In [27]:
Agentstate

{'messages': [], 'number': 1}

In [28]:
Agentstate["messages"].append("hi how are you?")

In [29]:
Agentstate["messages"].append("i hope everything is fine")

In [30]:
Agentstate

{'messages': ['hi how are you?', 'i hope everything is fine'], 'number': 1}

In [37]:
Agentstate['messages'][-1]

'i hope everything is fine'

### this agentstate class you need to inside the stategraph

# AIMessage
# HumanMessage
# ToolMessage
....


# Base Class - BaseMessage

In [34]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

In [35]:
state = {"messages": ["hi"]}

In [38]:
def function_1(state:AgentState):
    
    question=state["messages"][-1]
    
    print("Question",question)
    
    template="""
    Your task is to classify the given user query into one of the following categories: [USA,Not Related]. 
    Only respond with the category name and nothing else.

    User query: {question}
    {format_instructions}
    """
    
    prompt= PromptTemplate(
        template=template,
        input_variable=["question"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    
    chain= prompt | model | parser
    
    response = chain.invoke({"question":question})
    
    print("Parsed response:", response)
    
    return {"messages": [response.Topic]}

In [39]:
state={"messages":["what is a today weather?"]}

In [40]:
state_resp = function_1(state)
state_resp

Question what is a today weather?
Parsed response: Topic='Not Related' Reasoning="The query about today's weather does not specifically indicate a connection to the USA; it is a general question about weather."


{'messages': ['Not Related']}

In [41]:
state={"messages":["what is a GDP of usa??"]}

In [42]:
state_resp = function_1(state)
state_resp

Question what is a GDP of usa??
Parsed response: Topic='USA' Reasoning='The user query specifically asks about the GDP of the USA, which is directly related to the United States.'


{'messages': ['USA']}

In [43]:
def router(state:AgentState):
    print("-> ROUTER ->")
    
    last_message=state["messages"][-1]
    print("last_message:", last_message)
    
    if "usa" in last_message.lower():
        return "RAG Call"
    else:
        return "LLM Call"